# 7. 指示に従うためのファインチューニング

## 7.1 インストラクションチューニング

インストラクションチューニングを行うために，指示データセットを作る．

## 7.2 教師ありインストラクションチューニングのためのデータセットの準備

指示と応答のペア 1100 セットが入ったデータセットをダウンロードする．

In [71]:
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()
        
    with open(file_path, "r") as file:
        data = json.load(file)
    
    return data

file_path = "src/instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


例えばこんなデータが入っている

In [72]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [73]:
print("Another example entry:\n", data[999])

Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


入力と出力のペアが明示的に指定されたデータセットを使う．今回は上の JSON である．これを LLM 用にフォーマットする．
- Alpaca :指示の明記． md 形式でセクション 
- Phi-3 : トークンでセクションのみ (Microsoft)

今回は Alpaca 形式を採用する．

In [74]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describe a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    
    return instruction_text + input_text

In [75]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input+ desired_response)

Below is an instruction that describe a task.Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [76]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input+ desired_response)

Below is an instruction that describe a task.Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


データセットを訓練データセット (85%)，検証データセット (10%)，テストデータセット (5%) に分ける．

In [77]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion+test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
validation set length: 55
Test set length: 110


## 7.3 データを訓練バッチにまとめる

データをバッチ化する．デフォルトの `collate` 関数で楽する．
- `collate` : 個々のデータサンプルを 1 つのバッチにまとめる関数

**バッチ構築プロセス**
1. 入力を指示，応答テンプレートに変更
2. トークン ID 化
3. パディング (50256) して同じ長さに調整
4. シフトしてパディングトークンを追加したターゲットトークン ID を作成 (<-- 何故こんなことをするかは不明)
5. 特定のパディングトークンを -100 で置き換えて訓練の誤差計算から除外

In [78]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self,data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
    
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

複数の訓練サンプルをバッチにまとめて訓練を高速化する．`<|endoftext|>`トークン (50256) を追加する

In [79]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [80]:
# DataLoader に統合することを目的としている
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch) # バッチ内の最大長さを取得する
    inputs_lst = []
    
    for item in batch: # 入力のパディングと準備
        new_item = item.copy()
        new_item += [pad_token_id]
        
        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1]) # padded に追加した余計なパディングトークンを削除
        inputs_lst.append(inputs)
    
    return torch.stack(inputs_lst).to(device) # 入力のリストをテンソルに変換し，ターゲットデバイスに転送

（書籍通りに変数代入せずに直接返す関数にした）

In [81]:
# 使用例
inputs_1 = [0, 1, 2, 3, 4]
inputs_2= [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


ターゲットトークン ID は右に一つシフトしたものであり，これを作成する．

In [82]:
def custom_collate_draft_2(batch, pad_token_id=50256, device = "cpu"):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, target_lst = [], []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] * 
            (batch_max_length - len(new_item))
            )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        target_lst.append(targets)
    
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor, targets_tensor
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


ちゃんとシフトできてそうである．

各バッチでパディング成分が続く．最初だけ 50256 にしてそれ以降は-100 にするようにして collate を完成させる．

In [83]:
def custom_collate_fn(batch, pad_token_id = 50256, ignore_index = -100,
                      allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, target_lst = [], []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        
        padded = ( # シーケンスを max_length までパティング
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze() # tensor.squeeze() 成分数 1 の次元を削除 (A \times 1 \times B) -> (A \times B)
        if indices.numel() > 1: # 要素数 (A \times B \times C) -> |A| \times |B| \times |C|
            targets[indices[1:]] = ignore_index # 該当箇所に -100 で置き換える
        
        if allowed_max_length is not None: # 必要に応じてシーケンスの最大の長さで切り捨て
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        target_lst.append(targets)
    
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor, targets_tensor

In [84]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


-100 成分は，交差エントロピー関数計算時に無視される

In [85]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5]]
)
targets_1 = torch.tensor([0, 1])
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


In [86]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


In [87]:
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_ 3:", loss_1==loss_3)

tensor(1.1269)
loss_1 == loss_ 3: tensor(True)


パディングトークンをマスクすることに加えて，指示トークンをマスクすることも普遍的だが "Instcution Tuning With Loss Over Instruction" (Shi, etc 2024) では指示をマスクしない方がいいという主張もある．

## 7.4 指示データセット用のデータローダーを作成する

これまでターゲットデバイスへのデータ転送を訓練ループ内で行ってきたが、collate 関数の一部にしてしまうことでモデルの訓練中に GPU がブロック(?)されるのを防ぐ

In [88]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

Device : cuda


In [89]:
from functools import partial

custmized_collate_fn = partial( # 引数を次のように固定
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [90]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn = custmized_collate_fn,
    shuffle = True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custmized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custmized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [91]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 60]) torch.Size([8, 60])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 64]) torch.Size([8, 64])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 74]) torch.Size([8, 74])
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 78]) torch.Size([8, 78])
torch.Size([8, 70]) torch.Size([8, 70])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 82]) torch.Size([8, 82])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 70]) torch.Size([8, 70])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 64]) torch.Size([8, 64])
torch.Size([8, 67]) torch.

## 7.5 事前学習済みの LLM を読み込む

`gpt2-medium` を使う

In [92]:
from src.gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size" : 50257,
    "context_length" : 1024,
    "drop_rate" : 0.0,
    "qkv_bias" : True
}

model_configs = {
    "gpt2-small (124M)" : {"emb_dim":768, "n_layers" : 12, "n_heads" : 12},
    "gpt2-medium (355M)" : {"emb_dim":1024, "n_layers" : 24, "n_heads" : 16},
    "gpt2-large (774M)" : {"emb_dim":7280, "n_layers" : 36, "n_heads" : 20},
    "gpt2-xl (1558M)" : {"emb_dim":1600, "n_layers" : 48, "n_heads" : 25}
}

CHOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOSE_MODEL])

model_size = CHOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

File already exists and is up-to-date: gpt2/355M/checkpoint
File already exists and is up-to-date: gpt2/355M/encoder.json
File already exists and is up-to-date: gpt2/355M/hparams.json
File already exists and is up-to-date: gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/355M/model.ckpt.index
File already exists and is up-to-date: gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: gpt2/355M/vocab.bpe


GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

ダウンロードしたばかりのモデルでどれくらい指示追従能力があるのか検証。まずは指示内容を確認する。

In [ ]:
torch.manual_seed(123)

input_text = format_input(val_data[0])
print(input_text)

In [ ]:
from previous_chapters import generate, text_to_token_ids, token_ids_to_text
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)
generate_text = token_ids_to_text(token_ids, tokenizer)

In [ ]:
response_text = generate_text[len(input_text):].strip()
print(response_text)

まだ学習していないのででたらめな回答が返ってくる

## 7.6 指示データでの LLM のファインチューニング

In [ ]:
from previous_chapters import (
    calc_loss_loader,
    train_model_simple
)

model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=5
    )
    val_loss = calc_loss_loader(
        val_loader, model, device, num_batches=5
    )

print('Training loss :', train_loss)
print("Validation loss :", val_loss)

In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)